# Autoencoders & Variational Autoencoders (VAE) Notebook

> Hands-on Build It and Exercises.

## Build It

`code/main.py` implements a tiny VAE without numpy or torch. Input is 8-dimensional synthetic data drawn from a 2-component Gaussian mixture in 8-D. Encoder and decoder are single hidden-layer MLPs. We implement tanh activation, forward pass, loss, and a hand-written backward pass. Not production — pedagogy.

### Step 1: encoder forward

In [ ]:
```python

def encode(x, enc):

    h = tanh(add(matmul(enc["W1"], x), enc["b1"]))

    mu = add(matmul(enc["W_mu"], h), enc["b_mu"])

    log_sigma2 = add(matmul(enc["W_sig"], h), enc["b_sig"])

    return mu, log_sigma2

In [ ]:
```

`log σ²` instead of `σ` so the network output is unconstrained (softplus of σ is a trap — gradients die at σ ≈ 0).

### Step 2: reparameterize and decode

In [ ]:
```python

def reparameterize(mu, log_sigma2, rng):

    eps = [rng.gauss(0, 1) for _ in mu]

    sigma = [math.exp(0.5 * lv) for lv in log_sigma2]

    return [m + s * e for m, s, e in zip(mu, sigma, eps)]

def decode(z, dec):

    h = tanh(add(matmul(dec["W1"], z), dec["b1"]))

    return add(matmul(dec["W_out"], h), dec["b_out"])

In [ ]:
```

### Step 3: the ELBO

In [ ]:
```python

def elbo(x, x_hat, mu, log_sigma2, beta=1.0):

    recon = sum((a - b) ** 2 for a, b in zip(x, x_hat))

    kl = 0.5 * sum(math.exp(lv) + m * m - lv - 1 for m, lv in zip(mu, log_sigma2))

    return recon + beta * kl, recon, kl

In [ ]:
```

Exact closed-form KL because both distributions are Gaussian. Do not integrate numerically. People still ship code with monte-carlo KL estimates in 2026 — it is 3x slower for no reason.

### Step 4: generate

In [ ]:
```python

def sample(dec, z_dim, rng):

    z = [rng.gauss(0, 1) for _ in range(z_dim)]

    return decode(z, dec)

In [ ]:
```

That is the generative model. Five lines.

## Exercises

In [ ]:
1. **Easy.** Change `β` in `code/main.py` to `0.01`, `0.1`, `1.0`, `5.0`. Record the final reconstruction MSE and KL. Which β is Pareto-best for your synthetic data?
2. **Medium.** Replace the Gaussian decoder likelihood with a Bernoulli likelihood (cross-entropy loss). Compare sample quality on a binarized version of the same synthetic data.
3. **Hard.** Extend `code/main.py` into a mini VQ-VAE: replace the continuous `z` with a nearest-neighbour lookup in a codebook of K=32 entries. Compare reconstruction MSE and report how many codebook entries get used (codebook collapse is real).